# DWBI Week 5 — The ETL Process

**Ali Raza** · `su92-bsdsm-f23-018` · **BSDS-7A** · Superior University

PakFreight's Week-2 merge rebuilt as a pipeline that reruns: 37 rows land, 35 append, 34 load — 578,650 kg and PKR 4,808,000, exactly. Every number below is computed from `data/source/`, `data/dims/` or `data/night2/` in the notebook itself; nothing is typed in as a constant except the assertion target it is checked against.

## T4 — Row arithmetic (15 marks)

Every wrong total this week is 578,650 plus or minus something specific, and
the something tells you which step failed. The ladder is filled from the files,
not from the slides — including the road rung, which is the one that catches
people.

In [1]:

from pathlib import Path
import pandas as pd

BASE = Path("data"); SRC = BASE / "source"; DIMS = BASE / "dims"
CONTRACT = ["ShipmentID", "ShipDate", "OriginRaw", "DestRaw",
            "WeightKG", "FreightCostPKR", "StatusRaw"]
RENAME = {
    "marine": {"Shipment_ID": "ShipmentID", "Ship_Date": "ShipDate",
               "Origin_Port": "OriginRaw", "Dest_Port": "DestRaw",
               "Weight_KG": "WeightKG", "Cost_PKR": "FreightCostPKR",
               "Status": "StatusRaw"},
    "road": {"trip_id": "ShipmentID", "trip_date": "ShipDate",
             "from_city": "OriginRaw", "to_city": "DestRaw",
             "weight_kg": "WeightKG", "cost_pkr": "FreightCostPKR",
             "status": "StatusRaw"},
    "rail": {"Consignment No": "ShipmentID", "Dispatch Date": "ShipDate",
             "From Station": "OriginRaw", "To Station": "DestRaw",
             "Weight (Tons)": "WeightTons",
             "Freight Charges (PKR)": "FreightCostPKR", "Status": "StatusRaw"},
}
MODE = {"marine": 1, "rail": 2, "road": 3}

def extract(src, name):
    if name == "rail":
        return pd.read_excel(src / "rail_shipments.xlsx", sheet_name="Rail", skiprows=2)
    return pd.read_csv(src / f"{name}_shipments.csv")

def transform(df, name):
    df = df.rename(columns=RENAME[name])
    ids = df.ShipmentID.astype("string").str.strip()
    df = df[ids.notna() & (ids != "") & (ids.str.upper() != "TOTAL")].copy()
    for c in ("OriginRaw", "DestRaw"):
        df[c] = df[c].astype("string").str.strip()
    df["StatusRaw"] = (df.StatusRaw.astype("string")
                       .str.replace(r"\s+", " ", regex=True)
                       .str.strip().str.title())
    if name == "rail":
        df["ShipmentID"] = "RL-" + df.ShipmentID.astype(int).astype(str)
        df["WeightKG"] = df.WeightTons * 1000
        df = df.drop(columns=["WeightTons"])
        df["ShipDate"] = pd.to_datetime(df.ShipDate)
    else:
        df["ShipmentID"] = df.ShipmentID.astype("string").str.strip().str.upper()
        fmt = "%d/%m/%Y" if name == "marine" else "%Y-%m-%d"
        df["ShipDate"] = pd.to_datetime(df.ShipDate, format=fmt)
    df["WeightKG"] = df.WeightKG.astype("int64")
    df["FreightCostPKR"] = df.FreightCostPKR.astype("int64")
    df["ModeKey"] = MODE[name]
    return df[CONTRACT + ["ModeKey"]]

def alias_lookup():
    dim = pd.read_csv(DIMS / "DimCity.csv")
    out = (dim[["CityKey", "RawAliases"]]
           .assign(RawAliases=lambda d: d.RawAliases.str.split(" | ", regex=False))
           .explode("RawAliases"))
    out["RawAliases"] = out.RawAliases.str.strip()
    out = out[out.RawAliases.notna() & (out.RawAliases != "")]
    return out.rename(columns={"RawAliases": "RawSpelling"})

def pipeline(src=SRC):
    parts = [transform(extract(src, nm), nm) for nm in ("marine", "rail", "road")]
    fact = pd.concat(parts, ignore_index=True)
    appended = len(fact)
    fact = fact.drop_duplicates(subset=["ShipmentID"])
    alias = alias_lookup()
    for raw, key in (("OriginRaw", "OriginCityKey"), ("DestRaw", "DestCityKey")):
        fact = fact.merge(alias.rename(columns={"RawSpelling": raw, "CityKey": key}),
                          on=raw, how="left")
        fact[key] = fact[key].fillna(-1).astype("int64")
    return fact.reset_index(drop=True), appended, parts

fact, appended, parts = pipeline()
marine, rail, road = parts
road_raw = pd.read_csv("data/source/road_shipments.csv")
road_landed = len(road_raw)
road_naive = pd.to_numeric(road_raw.weight_kg, errors="coerce").sum()
print(f"marine: {len(marine)} rows, {marine.WeightKG.sum():,} kg, {marine.FreightCostPKR.sum():,} PKR")
print(f"rail:   {len(rail)} rows, {rail.WeightKG.sum():,} kg, {rail.FreightCostPKR.sum():,} PKR")
print(f"road landed: {road_landed} rows; naive sum over all 15 landed rows: {road_naive:,.0f} kg")
print(f"road surviving (13 real rows, duplicate inside): {len(road)} rows, {road.WeightKG.sum():,} kg")
print(f"appended: {appended}; loaded: {len(fact)}, {fact.WeightKG.sum():,} kg, {fact.FreightCostPKR.sum():,} PKR")
assert (len(marine), marine.WeightKG.sum()) == (12, 206500)
assert (len(rail), rail.WeightKG.sum()) == (10, 295250)
assert (len(road), road.WeightKG.sum()) == (13, 83000)
assert (appended, len(fact), fact.WeightKG.sum(), fact.FreightCostPKR.sum()) == (35, 34, 578650, 4808000)

marine: 12 rows, 206,500 kg, 1,618,000 PKR
rail:   10 rows, 295,250 kg, 2,585,000 PKR
road landed: 15 rows; naive sum over all 15 landed rows: 166,000 kg
road surviving (13 real rows, duplicate inside): 13 rows, 83,000 kg
appended: 35; loaded: 34, 578,650 kg, 4,808,000 PKR


**Analysis.** The ladder's arithmetic is now computed, not asserted from memory: marine 12 / 206,500, rail 10 / 295,250 after the tons conversion, and road 13 surviving rows at 83,000 kg with the RD-505 duplicate still inside. The naive sum over all 15 landed road rows is 166,000 kg — exactly double — because the TOTAL footer repeats the sum of the rows above it, which is the quantitative reason the footer filter must run before any aggregation.

![Row ladder](charts/t4_ladder.png)

### Part A · The ladder

| Stage | Rows | SUM(WeightKG) | SUM(FreightCostPKR) | What happens here |
|---|---|---|---|---|
| marine, as extracted | 12 | 206,500 | 1,618,000 | Clean file — no junk lines, nothing dropped. |
| rail, as extracted | 10 | 295,250 | 2,585,000 | 10 rows, after the two title rows are skipped; weights already converted from tons (295.25 t × 1,000). |
| road, as extracted | 15 landed → 13 survive | 83,000 | 652,000 | 15 rows land; the blank line and the TOTAL footer go. The 83,000 still contains the RD-505 duplicate — which is why the road rung and the loaded rung differ by 6,100 kg in weight terms once the duplicate is removed, not by nothing. Note the trap: summing all 15 landed rows naively gives 166,000 kg, because the TOTAL footer *is* the sum of the rows above it. |
| raw rows landed in staging | 37 | — | — | What the extract log must say, junk included: 12 + 10 + 15. |
| appended | 35 | 584,750 | 4,855,000 | 37 less the blank line and the TOTAL footer. |
| loaded into FactShipment | 34 | **578,650** | **4,808,000** | After duplicate RD-505 is removed: 584,750 − 6,100 = 578,650; 4,855,000 − 47,000 = 4,808,000. |

### Part B · Read the gap

| What you see | Rows | The arithmetic | Gap vs 578,650 | Missing/misplaced step | Cause and fix |
|---|---|---|---|---|---|
| SUM(WeightKG) = 283,695.25 kg | 34 | 206,500 + 76,900 + 295.25 | −294,954.75 | ConvertedTonsToKG missing, or run after the append | Rail tons were never multiplied by 1,000. Convert per source, before the append, so check 2 reads 295,250 on its own. |
| SUM(WeightKG) = 584,750 kg | 35 | 578,650 + 6,100 | +6,100 | RemovedDuplicates missing, or run on the whole row before trimming | Duplicate RD-505 survived (6,100 kg, PKR 47,000). Dedup on ShipmentID, after trim and uppercase. |
| SUM(WeightKG) = 661,650 kg | 35 | 578,650 + 83,000 | +83,000 | RemovedTotalRow missing | The road TOTAL footer loaded as a shipment. Filter `ShipmentID = TOTAL` out in the road query, as its own named step. |

### Two more symptoms — and neither moves a total

**Headers read `Column1, Column2, …`.** Cause: on the rail query, headers were
promoted *before* the two title rows were skipped, so the title text became data and Power Query
invented column names. Fix: Remove Top Rows 2, *then* Use First Row as Headers — shape in that
order, always. This is the one symptom that is loud: the next step cannot find its columns.

**FactShipment has 35 rows but both totals are exactly right.** Cause: the blank road row
loaded. It carries no weight and no cost, so it moves `COUNTROWS` and neither total — and it
appears as `(Blank)` in every slicer. This is the entire reason check 5 (row count) exists
separately from checks 7 and 8 (totals): a row count is not a total, and a total is not a row
count, and a pipeline that only reconciles totals will bless a table with a phantom row in it.